# Multiview classification: Label Studio + FiftyOne

This notebook creates one logical sample from multiple image slices, visualizes the groups in FiftyOne, exports one Label Studio task per group, and imports the resulting group-level class back into every slice.

The default run is self-contained and creates synthetic images. Set `GENERATE_DEMO_DATA = False` and provide an absolute-path manifest to use real data.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.parse import quote

import fiftyone as fo
from fiftyone import ViewField as F
from PIL import Image, ImageDraw

# Launch Jupyter from the repository root, as shown in the README.
REPO_ROOT = Path.cwd().resolve()
EXAMPLE_ROOT = REPO_ROOT / "examples/multiview-classification"
if not EXAMPLE_ROOT.is_dir():
    raise RuntimeError(
        "EXAMPLE_ROOT was not found. Launch Jupyter from the repository root "
        "or set REPO_ROOT explicitly."
    )

DATASET_NAME = "multiview-classification-demo"
RECREATE_DATASET = True
GENERATE_DEMO_DATA = True
MANIFEST_PATH = EXAMPLE_ROOT / "fiftyone/manifest.example.json"
DEMO_MEDIA_DIR = EXAMPLE_ROOT / "fiftyone/generated-media"
ARTIFACT_DIR = EXAMPLE_ROOT / "fiftyone/artifacts"
LABEL_SPACE = ("ok", "misaligned", "damaged")
DEFAULT_SLICE = "front"

print("FiftyOne:", fo.__version__)
print("Example root:", EXAMPLE_ROOT)

## Build or load the logical-sample manifest

The manifest has one row per logical sample. Every image entry has a stable view name and an absolute filepath. Validation is intentionally strict: missing files, duplicate IDs/views, unknown labels, and a missing default slice are errors.

In [ ]:
def generate_demo_manifest(media_dir: Path) -> list[dict[str, object]]:
    media_dir.mkdir(parents=True, exist_ok=True)
    labels = ("ok", "misaligned", "damaged", "ok", "damaged", "misaligned")
    views = ("front", "left", "right")
    colors = {"front": "#2563eb", "left": "#16a34a", "right": "#ea580c"}
    records: list[dict[str, object]] = []

    for index, label in enumerate(labels, start=1):
        sample_id = f"part-{index:04d}"
        images: list[dict[str, str]] = []
        for view_name in views:
            image_dir = media_dir / sample_id
            image_dir.mkdir(parents=True, exist_ok=True)
            image_path = (image_dir / f"{view_name}.png").resolve()
            image = Image.new("RGB", (960, 640), colors[view_name])
            draw = ImageDraw.Draw(image)
            draw.text((48, 48), f"{sample_id} / {view_name}", fill="white")
            draw.text((48, 96), f"logical class: {label}", fill="white")
            image.save(image_path)
            images.append({"view": view_name, "filepath": str(image_path)})

        records.append({"sample_id": sample_id, "label": label, "images": images})

    return records


def load_manifest(path: Path) -> list[dict[str, object]]:
    if not path.is_file():
        raise FileNotFoundError(f"Manifest does not exist: {path}")
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, list):
        raise ValueError("Manifest root must be a JSON array")
    return payload


def validate_manifest(records: list[dict[str, object]]) -> None:
    seen_sample_ids: set[str] = set()
    for record in records:
        sample_id = record.get("sample_id")
        if not isinstance(sample_id, str) or not sample_id:
            raise ValueError(f"Invalid sample_id: {sample_id!r}")
        if sample_id in seen_sample_ids:
            raise ValueError(f"Duplicate sample_id: {sample_id}")
        seen_sample_ids.add(sample_id)

        label = record.get("label")
        if label is not None and label not in LABEL_SPACE:
            raise ValueError(f"Unknown label for {sample_id}: {label!r}")

        images = record.get("images")
        if not isinstance(images, list) or not images:
            raise ValueError(f"{sample_id} must contain at least one image")

        seen_views: set[str] = set()
        for image in images:
            if not isinstance(image, dict):
                raise ValueError(f"Invalid image entry for {sample_id}: {image!r}")
            view_name = image.get("view")
            if not isinstance(view_name, str) or not view_name:
                raise ValueError(f"Invalid view name for {sample_id}: {view_name!r}")
            if view_name in seen_views:
                raise ValueError(f"Duplicate ({sample_id}, {view_name}) image")
            seen_views.add(view_name)

            raw_path = image.get("filepath")
            if not isinstance(raw_path, str):
                raise ValueError(f"Invalid filepath for {sample_id}/{view_name}")
            image_path = Path(raw_path)
            if not image_path.is_absolute():
                raise ValueError(f"Filepath must be absolute: {image_path}")
            if not image_path.is_file():
                raise FileNotFoundError(f"Image does not exist: {image_path}")

        if DEFAULT_SLICE not in seen_views:
            raise ValueError(f"{sample_id} is missing default slice {DEFAULT_SLICE!r}")


records = (
    generate_demo_manifest(DEMO_MEDIA_DIR)
    if GENERATE_DEMO_DATA
    else load_manifest(MANIFEST_PATH.resolve())
)
validate_manifest(records)
print(f"Validated {len(records)} logical samples")

## Create a FiftyOne grouped dataset

FiftyOne stores each image as a physical sample. A single `fo.Group` connects all views of one logical sample. The classification is copied to each slice so filters behave the same regardless of the active slice.

In [ ]:
if fo.dataset_exists(DATASET_NAME):
    if not RECREATE_DATASET:
        raise RuntimeError(
            f"Dataset {DATASET_NAME!r} already exists; set RECREATE_DATASET=True "
            "or choose another name"
        )
    fo.delete_dataset(DATASET_NAME)

dataset = fo.Dataset(DATASET_NAME)
dataset.persistent = True
physical_samples: list[fo.Sample] = []

for record in records:
    sample_id = str(record["sample_id"])
    label = record.get("label")
    group = fo.Group()
    for image in record["images"]:
        view_name = str(image["view"])
        sample = fo.Sample(filepath=str(image["filepath"]), tags=["multiview"])
        sample["group"] = group.element(view_name)
        sample["logical_sample_id"] = sample_id
        sample["view_name"] = view_name
        if label is not None:
            sample["ground_truth"] = fo.Classification(label=str(label))
        physical_samples.append(sample)

dataset.add_samples(physical_samples)
dataset.default_group_slice = DEFAULT_SLICE
dataset.save()

print(dataset)
print("Group slices:", dataset.group_slices)
print("Logical samples:", len(records))
print("Physical image samples:", len(dataset.select_group_slices()))

In [ ]:
def assert_group_consistency(
    source: fo.Dataset,
    label_field: str,
    required_slices: set[str] | None = None,
) -> None:
    required_slices = required_slices or {DEFAULT_SLICE}
    all_slices = source.select_group_slices()
    for logical_sample_id in all_slices.distinct("logical_sample_id"):
        group_view = all_slices.match(F("logical_sample_id") == logical_sample_id)
        slices = set(group_view.values("group.name"))
        missing_slices = required_slices - slices
        if missing_slices:
            raise AssertionError(
                f"{logical_sample_id} is missing slices: {sorted(missing_slices)}"
            )
        labels = {label for label in group_view.values(f"{label_field}.label") if label}
        if len(labels) > 1:
            raise AssertionError(
                f"{logical_sample_id} has inconsistent {label_field}: {sorted(labels)}"
            )


assert_group_consistency(dataset, "ground_truth", set(dataset.group_slices))
print("All groups and labels are consistent")

## Visualize and query groups

The App's group view lets you switch between slices while preserving the logical group. The grid initially uses `front` because it is the configured default slice.

In [ ]:
session = fo.launch_app(dataset)
print("FiftyOne App launched. In a script, call session.wait() to keep it open.")
# session.wait()

In [ ]:
# Count each logical sample once by selecting exactly one slice.
logical_view = dataset.select_group_slices(DEFAULT_SLICE)
for label in LABEL_SPACE:
    count = len(logical_view.match(F("ground_truth.label") == label))
    print(f"{label:>12}: {count}")

damaged_groups = logical_view.match(F("ground_truth.label") == "damaged")
session.view = damaged_groups
print("App view now shows damaged logical samples")

## Add predictions and evaluate once per logical sample

Predictions are copied across slices for consistent browsing. Evaluation runs only on `front`, preventing a three-view sample from being counted three times.

In [ ]:
prediction_by_sample_id = {
    "part-0001": ("ok", 0.98),
    "part-0002": ("damaged", 0.58),
    "part-0003": ("damaged", 0.93),
    "part-0004": ("ok", 0.88),
    "part-0005": ("misaligned", 0.54),
    "part-0006": ("misaligned", 0.96),
}

all_slices = dataset.select_group_slices()
known_ids = set(all_slices.distinct("logical_sample_id"))
if set(prediction_by_sample_id) != known_ids:
    raise ValueError("prediction_by_sample_id must cover every logical sample exactly")

for logical_sample_id, (label, confidence) in prediction_by_sample_id.items():
    if label not in LABEL_SPACE:
        raise ValueError(f"Unknown prediction label: {label}")
    group_view = all_slices.match(F("logical_sample_id") == logical_sample_id)
    for sample in group_view:
        sample["predictions"] = fo.Classification(label=label, confidence=confidence)
        sample.save()

assert_group_consistency(dataset, "predictions", set(dataset.group_slices))
evaluation_view = dataset.select_group_slices(DEFAULT_SLICE)
results = evaluation_view.evaluate_classifications(
    "predictions", gt_field="ground_truth", eval_key="logical_eval"
)
results.print_report()

## Export one Label Studio task per group

This creates payloads for `variable-views.xml` and `three-view-grid.xml`. Local file URLs are generated relative to an explicitly configured Label Studio document root.

In [ ]:
LABEL_STUDIO_DOCUMENT_ROOT = DEMO_MEDIA_DIR.resolve()
INCLUDE_PREANNOTATIONS = False


def label_studio_local_url(filepath: str, document_root: Path) -> str:
    path = Path(filepath).resolve()
    root = document_root.resolve()
    try:
        relative = path.relative_to(root)
    except ValueError as exc:
        raise ValueError(
            f"{path} is outside Label Studio document root {root}"
        ) from exc
    return f"/data/local-files/?d={quote(relative.as_posix())}"


def preannotation(label: str, to_name: str) -> list[dict[str, object]]:
    return [
        {
            "model_version": "manifest-bootstrap",
            "result": [
                {
                    "from_name": "classification",
                    "to_name": to_name,
                    "type": "choices",
                    "value": {"choices": [label]},
                }
            ],
        }
    ]


def build_variable_task(record: dict[str, object]) -> dict[str, object]:
    images = record["images"]
    task: dict[str, object] = {
        "data": {
            "sample_id": record["sample_id"],
            "images": [
                label_studio_local_url(
                    str(image["filepath"]), LABEL_STUDIO_DOCUMENT_ROOT
                )
                for image in images
            ],
            "view_names": [image["view"] for image in images],
        }
    }
    if INCLUDE_PREANNOTATIONS and record.get("label") is not None:
        task["predictions"] = preannotation(str(record["label"]), "images")
    return task


def build_three_view_task(record: dict[str, object]) -> dict[str, object]:
    paths_by_view = {
        str(image["view"]): label_studio_local_url(
            str(image["filepath"]), LABEL_STUDIO_DOCUMENT_ROOT
        )
        for image in record["images"]
    }
    required = {"front", "left", "right"}
    if set(paths_by_view) != required:
        raise ValueError(
            f"Three-view export requires exactly {sorted(required)}; got {sorted(paths_by_view)}"
        )
    task: dict[str, object] = {
        "data": {"sample_id": record["sample_id"], **paths_by_view}
    }
    if INCLUDE_PREANNOTATIONS and record.get("label") is not None:
        task["predictions"] = preannotation(str(record["label"]), "front")
    return task


ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
variable_tasks = [build_variable_task(record) for record in records]
grid_tasks = [build_three_view_task(record) for record in records]
variable_path = ARTIFACT_DIR / "tasks.variable.generated.json"
grid_path = ARTIFACT_DIR / "tasks.three-view.generated.json"
variable_path.write_text(json.dumps(variable_tasks, indent=2), encoding="utf-8")
grid_path.write_text(json.dumps(grid_tasks, indent=2), encoding="utf-8")
print("Wrote:", variable_path)
print("Wrote:", grid_path)

Start Label Studio with `LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT` set to the printed document root, apply the matching XML config, and import a generated task file. After annotation, export JSON and use the next cell to sync it.

In [ ]:
print("export LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED=true")
print(f"export LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT={LABEL_STUDIO_DOCUMENT_ROOT}")

## Import completed Label Studio annotations

The importer requires exactly one non-cancelled annotation and one single-valued `classification` result per task. It updates all slices in the corresponding FiftyOne group. Set `APPLY_LABEL_STUDIO_EXPORT = True` only after choosing the intended export file.

In [ ]:
LS_EXPORT_PATH = EXAMPLE_ROOT / "label-studio/export.example.json"
APPLY_LABEL_STUDIO_EXPORT = False


def extract_completed_label(task: dict[str, object]) -> tuple[str, str]:
    data = task.get("data")
    if not isinstance(data, dict) or not isinstance(data.get("sample_id"), str):
        raise ValueError(f"Task {task.get('id')} has no string data.sample_id")
    sample_id = data["sample_id"]

    annotations = [
        annotation
        for annotation in task.get("annotations", [])
        if not annotation.get("was_cancelled", False)
    ]
    if len(annotations) != 1:
        raise ValueError(
            f"Task {task.get('id')} for {sample_id} must have exactly one completed annotation"
        )

    matches = [
        result
        for result in annotations[0].get("result", [])
        if result.get("from_name") == "classification"
        and result.get("type") == "choices"
    ]
    if len(matches) != 1:
        raise ValueError(f"Task {task.get('id')} must have one classification result")
    choices = matches[0].get("value", {}).get("choices", [])
    if len(choices) != 1 or choices[0] not in LABEL_SPACE:
        raise ValueError(f"Task {task.get('id')} has invalid choices: {choices!r}")
    return sample_id, choices[0]


def sync_label_studio_export(source: fo.Dataset, export_path: Path) -> int:
    payload = json.loads(export_path.read_text(encoding="utf-8"))
    if not isinstance(payload, list):
        raise ValueError("Label Studio export root must be a JSON array")

    labels_by_sample_id = dict(extract_completed_label(task) for task in payload)
    if len(labels_by_sample_id) != len(payload):
        raise ValueError("Label Studio export contains duplicate sample_id values")

    all_slices = source.select_group_slices()
    known_ids = set(all_slices.distinct("logical_sample_id"))
    unknown_ids = set(labels_by_sample_id) - known_ids
    if unknown_ids:
        raise ValueError(f"Export contains unknown sample IDs: {sorted(unknown_ids)}")

    updated_images = 0
    for sample_id, label in labels_by_sample_id.items():
        group_view = all_slices.match(F("logical_sample_id") == sample_id)
        for sample in group_view:
            sample["ground_truth"] = fo.Classification(label=label)
            sample.save()
            updated_images += 1

    assert_group_consistency(source, "ground_truth")
    return updated_images


if APPLY_LABEL_STUDIO_EXPORT:
    updated_images = sync_label_studio_export(dataset, LS_EXPORT_PATH.resolve())
    print(f"Updated {updated_images} physical images from {LS_EXPORT_PATH}")
    session.refresh()
else:
    print("Dry run: set APPLY_LABEL_STUDIO_EXPORT=True to apply the selected export")

## Optional: FiftyOne's native Label Studio backend for one slice

Use this when each selected image should become its own Label Studio task. It is intentionally limited to the default slice and is separate from the custom one-task-per-group export above. Credentials are read from `FIFTYONE_LABELSTUDIO_API_KEY`; the URL can be supplied through the example annotation config or `FIFTYONE_LABELSTUDIO_URL`.

In [ ]:
RUN_NATIVE_PER_SLICE_ANNOTATION = False

if RUN_NATIVE_PER_SLICE_ANNOTATION:
    unlabeled_front = dataset.select_group_slices(DEFAULT_SLICE).exists(
        "ground_truth", False
    )
    if len(unlabeled_front) == 0:
        raise RuntimeError("No unlabeled default-slice samples were selected")
    unlabeled_front.annotate(
        "front-slice-classification",
        backend="labelstudio",
        label_field="ground_truth",
        label_type="classification",
        classes=list(LABEL_SPACE),
        project_name="FiftyOne multiview front slice",
        launch_editor=True,
    )
else:
    print("Native per-slice annotation is disabled")